In [1]:
# The imports
import pandas as pd
import numpy as np

In [2]:
# Import the parsed dataframe to this notebook
data = pd.read_parquet("parsed_data.parquet")

In [3]:
# Create a "subset" dataFrame that points to the rows where at least one value is NA
na_mask = data.isna().any(axis=1)
missing_vals = data.loc[na_mask]

In [4]:
fig1_data = data.groupby(['Region', 'Month'])[['Discharges', 'Bed days']].sum()
fig1_data.reset_index(inplace=True) # To be able to group by the dates' month value in the next lines

# Create new columns that store the maximum and minimum bed days and discharge values reached in each month throughout the years
fig1_data["Discharge max monthly value"] = fig1_data.groupby(['Region', fig1_data['Month'].dt.month])['Discharges'].transform("max")
fig1_data["Discharge min monthly value"] = fig1_data.groupby(['Region', fig1_data['Month'].dt.month])['Discharges'].transform("min")
fig1_data["Bed days max monthly value"] = fig1_data.groupby(['Region', fig1_data['Month'].dt.month])['Bed days'].transform("max")
fig1_data["Bed days min monthly value"] = fig1_data.groupby(['Region', fig1_data['Month'].dt.month])['Bed days'].transform("min")


fig1_data

,Region,Month,Discharges,Bed days,Discharge max monthly value,Discharge min monthly value,Bed days max monthly value,Bed days min monthly value
0,Região de Saúde LVT,2015-01,25517.0,246678.0,26400.0,19629.0,246678.0,194367.0
1,Região de Saúde LVT,2015-02,49438.0,430676.0,50376.0,38380.0,442625.0,384324.0
2,Região de Saúde LVT,2015-03,75233.0,655194.0,77034.0,59579.0,660780.0,577235.0
3,Região de Saúde LVT,2015-04,99849.0,859224.0,100300.0,80860.0,873583.0,761352.0
4,Região de Saúde LVT,2015-05,124725.0,1066996.0,124905.0,98685.0,1091225.0,942481.0
...,...,...,...,...,...,...,...,...
670,Região de Saúde do Centro,2025-11,134242.0,1095037.0,153327.0,116810.0,1230995.0,1036240.0
671,Região de Saúde do Centro,2025-12,146205.0,1213784.0,167504.0,128343.0,1347886.0,1142249.0
672,Região de Saúde do Centro,2026-01,12314.0,107995.0,14590.0,10350.0,121890.0,96215.0
673,Região de Saúde do Centro,2026-02,23463.0,188448.0,28389.0,20023.0,234652.0,188448.0


In [5]:
fig2_data = data.groupby(['Region', 'Month'])[['Discharges', 'Bed days']].sum()
fig2_data.reset_index(inplace=True) # To be able to group by the dates' month value in the next lines

# Create a new column that stores the median bed days and discharge values reached in each month throughout the years
fig2_data["Discharge median monthly value"] = fig2_data.groupby(['Region', fig2_data['Month'].dt.month])['Discharges'].transform(np.median)
fig2_data["Bed days median monthly value"] = fig2_data.groupby(['Region', fig2_data['Month'].dt.month])['Bed days'].transform(np.median)

fig2_data = fig2_data.query('Month.dt.year == 2025')

In [6]:
fig3_data = (
    data
    .groupby(['Region', 'Specialty type'])  
    .resample('QE')                         
    [['Discharges', 'Bed days']]
    .sum()
)

fig3_data = fig3_data.swaplevel(2, 1)
fig3_data

Discharges  \
Region                    Month  Specialty type                        
Região de Saúde LVT       2015Q1 Especialidade Cirurgica     68145.0   
                          2015Q2 Especialidade Cirurgica    174373.0   
                          2015Q3 Especialidade Cirurgica    277308.0   
                          2015Q4 Especialidade Cirurgica    385311.0   
                          2016Q1 Especialidade Cirurgica     67486.0   
...                                                              ...   
Região de Saúde do Centro 2025Q1 Outras Camas                 3726.0   
                          2025Q2 Outras Camas                 9282.0   
                          2025Q3 Outras Camas                15027.0   
                          2025Q4 Outras Camas                20791.0   
                          2026Q1 Outras Camas                 3709.0   

                                                           Bed days  
Region                    Month  Specialty type                      
Região de Saúde LVT       2015Q1 Especialidade Cirurgica   457423.0  
                          2015Q2 Especialidade Cirurgica  1123569.0  
                          2015Q3 Especialidade Cirurgica  1783081.0  
                          2015Q4 Especialidade Cirurgica  2470840.0  
                          2016Q1 Especialidade Cirurgica   444152.0  
...                                                             ...  
Região de Saúde do Centro 2025Q1 Outras Camas               49706.0  
                          2025Q2 Outras Camas              120105.0  
                          2025Q3 Outras Camas              186597.0  
                          2025Q4 Outras Camas              264546.0  
                          2026Q1 Outras Camas               48655.0  

[675 rows x 2 columns]

In [ ]:
# Intermediate tables to help fetch the required values with more readable code syntax
surg_data = data[data["Specialty type"] == "Especialidade Cirurgica"]
med_data = data[data["Specialty type"] == "Especialidade Médica"]
other_data = data[data["Specialty type"] == "Outras Camas"]

annual_data = data.groupby("Region")[['Bed days', 'Discharges']].resample("YE").sum()
annual_data.reset_index(inplace = True)
annual_data.rename(columns = {'Month': "Year"}, inplace = True)

# Build the dataframe with the required values.
final_table_data = (   # Expression on multiple lines for enhanced readability; uses the concept of named aggregation, with plain tuples. Found the feature at https://pandas.pydata.org/docs/user_guide/groupby.html#named-aggregation
    data
    .groupby('Region')
    .agg(
        mean_bed_days=('Bed days', 'mean'),
        std_bed_days=('Bed days', 'std'),
        mean_discharges=('Discharges', 'mean'),
        std_discharges=('Discharges', 'std'),
    )
)

final_table_data['coef bed days'] = final_table_data['std_bed_days'] / final_table_data['mean_bed_days']
final_table_data['coef discharges'] = final_table_data['std_discharges'] / final_table_data['mean_discharges']

final_table_data['Surgical total bed days'] = surg_data.groupby("Region")["Bed days"].sum()
final_table_data['Medical total bed days'] = med_data.groupby("Region")["Bed days"].sum()
final_table_data['Other total bed days'] = other_data.groupby("Region")["Bed days"].sum()

final_table_data['Surgical total discharges'] = surg_data.groupby("Region")["Discharges"].sum()
final_table_data['Medical total discharges'] = med_data.groupby("Region")["Discharges"].sum()
final_table_data['Other total discharges'] = other_data.groupby("Region")["Discharges"].sum()

final_table_data['Surg bed days per discharge'] = final_table_data['Surgical total bed days'] / final_table_data['Surgical total discharges']
final_table_data['Med bed days per discharge'] = final_table_data['Medical total bed days'] / final_table_data['Medical total discharges']
final_table_data['Other bed days per discharge'] = final_table_data['Other total bed days'] / final_table_data['Other total discharges']

# final_table_data2 = annual_data.groupby('Region')[['Bed days', 'Discharges']].transform()

,Region,Year,Bed days,Discharges
